# 20_05_FashionMNIST_を_3_エポック学習す

コード：`ch20/blocks/20_05_FashionMNIST_を_3_エポック学習す.ipynb`

本文のコードをコードセルで直接実行します。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 付録 A-5：学習と評価のループ（FashionMNIST・3 エポック・CPU）
import time, torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

torch.manual_seed(0)
train = DataLoader(datasets.FashionMNIST("data", train=True, download=True, transform=ToTensor()),
                   batch_size=64, shuffle=True)
test = DataLoader(datasets.FashionMNIST("data", train=False, download=True, transform=ToTensor()), batch_size=256)

model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
loss_fn = nn.CrossEntropyLoss()                           # ロジットと正解の番号から損失を出す（softmax を含む）
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

def train_one_epoch():
    model.train()                                         # 学習モード（Dropout などが効く）
    total = 0.0
    for xb, yb in train:
        opt.zero_grad()                                   # ① 前の勾配を消す
        loss = loss_fn(model(xb), yb)                     # ② 予測して ③ ずれを測る
        loss.backward()                                   # ④ 勾配を求める
        opt.step()                                        # ⑤ 少し動かす
        total += loss.item() * len(xb)
    return total / len(train.dataset)

@torch.no_grad()
def evaluate():
    model.eval()                                          # 評価モード
    correct = sum((model(xb).argmax(1) == yb).sum().item() for xb, yb in test)
    return correct / len(test.dataset)

print("学習前の正解率", round(evaluate(), 4))
for epoch in range(1, 4):
    t0 = time.time()
    loss = train_one_epoch()
    print(f"エポック {epoch}：訓練の損失 {loss:.4f}  テストの正解率 {evaluate():.4f}  （{time.time() - t0:.1f} 秒）")
torch.save(model.state_dict(), "mlp.pt")
